In [0]:
from pyspark.sql.functions import row_number, monotonically_increasing_id
from pyspark.sql import functions as F
from pyspark.sql import Window
import json

In [0]:
def pool_retrieved_results(spark, db, table_names, run_date, collect=False, add_row_num=True, num_shards=1):
    if collect:
        sdf = spark.sql("""
                        WITH all_retrieval_channels AS (
                            SELECT * from {db}.{geo_retrieval} where partition_date = '{run_date}'
                            UNION ALL
                            SELECT * FROM {db}.{follower_retrieval} where partition_date = '{run_date}'
                            UNION ALL
                            SELECT * FROM {db}.{i2i_retrieval} where partition_date = '{run_date}'
                            UNION ALL
                            SELECT * FROM {db}.{two_tower_retrieval} where partition_date = '{run_date}'
                        )
                        SELECT
                            user_id,
                            array_distinct(flatten(collect_list(post_id))) AS retrieved_post_ids
                        FROM all_retrieval_channels
                        GROUP BY user_id
                        """.format(db = db,
                                   i2i_retrieval = table_names["i2i_retrieval"],
                                   two_tower_retrieval = table_names["two_tower_retrieval"],
                                   geo_retrieval = table_names["geo_retrieval"],
                                   follower_retrieval = table_names["follower_retrieval"],
                                   run_date = run_date))
    else:
        sdf = spark.sql("""
                        SELECT DISTINCT user_id, post_id
                        FROM (
                            SELECT user_id, explode(post_id) as post_id from {db}.{geo_retrieval} where partition_date = '{run_date}'
                            UNION ALL
                            SELECT user_id, explode(post_id) as post_id FROM {db}.{follower_retrieval} where partition_date = '{run_date}'
                            UNION ALL
                            SELECT user_id, explode(post_id) as post_id FROM {db}.{i2i_retrieval} where partition_date = '{run_date}'
                            UNION ALL
                            SELECT user_id, explode(post_id) as post_id FROM {db}.{two_tower_retrieval} where partition_date = '{run_date}'
                        )
                        """.format(db = db,
                                   i2i_retrieval = table_names["i2i_retrieval"],
                                   two_tower_retrieval = table_names["two_tower_retrieval"],
                                   geo_retrieval = table_names["geo_retrieval"],
                                   follower_retrieval = table_names["follower_retrieval"],
                                   run_date = run_date))
    if add_row_num:
        sdf = sdf.withColumn("row_id", row_number().over(Window.orderBy(monotonically_increasing_id())))
    shard_count = max(1, int(num_shards))
    if shard_count > 1:
        sdf = sdf.withColumn("shard_id", F.pmod(F.xxhash64("user_id"), F.lit(shard_count)))
    else:
        sdf = sdf.withColumn("shard_id", F.lit(0).cast("long"))
    return sdf

In [0]:
def get_user_post_interaction_retrieval_samples(spark, db, table_names, run_date, include_all_pos=True, to_pandas=False):
    sdf = spark.sql("""
                    -- user may have different interactions on different days
                    select upi.user_id, upi.post_id, upi.interaction, upi.partition_date
                    from {db}.{user_post_interaction} upi
                    inner join {db}.{all_retrieval_results} arr
                    on upi.user_id = arr.user_id
                    and upi.post_id = arr.post_id
                    where upi.partition_date <= '{run_date}'
                    and upi.interaction != 'publish' --exclude posts published by the user himself
                    """.format(db = db,
                               user_post_interaction = table_names["user_post_interaction"],
                               all_retrieval_results = table_names["all_retrieval_results"],
                               run_date = run_date))
    if include_all_pos:
        # Other than retrieved candidates, include all positive interaction examples
        sdf.createOrReplaceTempView("interaction_samples")
        sdf = spark.sql("""
                        select *
                        from interaction_samples
                        union
                        select *
                        from {db}.{user_post_interaction} upi
                        where upi.partition_date <= '{run_date}'
                        and upi.interaction not in ('impression','publish')
                        and upi.user_id in (select user_id from {db}.{user_attributes} where partition_date <= '{run_date}')
                        and upi.post_id in (select post_id from {db}.{post_attributes} where partition_date <= '{run_date}')
                        """.format(db = db,
                                   user_post_interaction = table_names["user_post_interaction"],
                                   user_attributes = table_names["user_attributes"],
                                   post_attributes = table_names["post_attributes"],
                                   run_date = run_date))
    return sdf.toPandas() if to_pandas else sdf

def get_user_post_dwell_time_retrieval_samples(spark, db, table_names, run_date, include_all_pos=True, to_pandas=False):
    sdf = spark.sql("""
                    -- user may have different dwell time on different days
                    select updt.user_id, updt.post_id, updt.dwell_time, updt.partition_date
                    from {db}.{user_post_dwell_time} updt
                    inner join {db}.{all_retrieval_results} arr
                    on updt.user_id = arr.user_id
                    and updt.post_id = arr.post_id
                    where updt.partition_date <= '{run_date}'
                    """.format(db = db,
                               user_post_dwell_time = table_names["user_post_dwell_time"],
                               all_retrieval_results = table_names["all_retrieval_results"],
                               run_date = run_date))
    if include_all_pos:
        # Other than retrieved candidates, include all positive dwell time examples
        sdf.createOrReplaceTempView("dwell_samples")
        sdf = spark.sql("""
                        select *
                        from dwell_samples
                        union
                        select *
                        from {db}.{user_post_dwell_time} updt
                        where updt.partition_date <= '{run_date}'
                        and updt.dwell_time > 0
                        and updt.user_id in (select user_id from {db}.{user_attributes} where partition_date <= '{run_date}')
                        and updt.post_id in (select post_id from {db}.{post_attributes} where partition_date <= '{run_date}')
                        """.format(db = db,
                                   user_post_dwell_time = table_names["user_post_dwell_time"],
                                   user_attributes = table_names["user_attributes"],
                                   post_attributes = table_names["post_attributes"],
                                   run_date = run_date))
    return sdf.toPandas() if to_pandas else sdf

In [0]:
# Get pipeline parameters
dbutils.widgets.text("ranking_num_shards", "1")
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))
ranking_num_shards = int(dbutils.widgets.get("ranking_num_shards") or "1")
print("Input parameter ranking_num_shards:{}".format(ranking_num_shards))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

In [0]:
sdf_all_ret = pool_retrieved_results(
    spark,
    db,
    table_names,
    run_date,
    collect=False,
    add_row_num=True,
    num_shards=ranking_num_shards,
)
sdf_all_ret.write.mode("overwrite").option("mergeSchema","true").saveAsTable(f"{db}.{table_names['all_retrieval_results']}")

In [0]:
sdf_interaction = get_user_post_interaction_retrieval_samples(spark, db, table_names, run_date, include_all_pos=True)
sdf_interaction.write.mode("overwrite").partitionBy("partition_date").saveAsTable(f"{db}.{table_names['interaction_retrieved_candidates']}")

In [0]:
sdf_dwell = get_user_post_dwell_time_retrieval_samples(spark, db, table_names, run_date, include_all_pos=True)
sdf_dwell.write.mode("overwrite").partitionBy("partition_date").saveAsTable(f"{db}.{table_names['dwell_retrieved_candidates']}")